# CSE598/EEE589 Project 05 — results and decision guide

Load exported runs, check that hardware and dtype can be compared, and draw measured figures only from successful rows. Untested choices stay marked untested. This notebook does not declare a winning algorithm.


## Configuration


In [ ]:
# Fresh Colab runtime: upload the project ZIP to /content and set ZIP_PATH.
# If the repository has a remote, set REPO_URL and the full REPO_COMMIT instead.
# Leave all three empty when this notebook is already running inside the project tree.
REPO_URL = ""
REPO_COMMIT = ""
ZIP_PATH = ""
PROJECT_DIR = ""

RESULTS_DIR = ""


## Setup


In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

def discover(start):
    root = Path(start).resolve()
    pool = [root, *root.parents]
    if root.is_dir():
        for child in root.iterdir():
            if child.is_dir() and child.name not in {".venv", ".git"}:
                pool.append(child)
                for grandchild in child.iterdir():
                    if grandchild.is_dir() and grandchild.name not in {".venv", ".git"}:
                        pool.append(grandchild)
    for candidate in pool:
        if (candidate / "src" / "spbench" / "__init__.py").is_file():
            return candidate
    return None

if ZIP_PATH:
    dest = Path("/content/long-context-attention-benchmark" if os.path.isdir("/content") else "colab_unzipped")
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(dest)
    project = discover(dest)
elif REPO_URL:
    if not REPO_COMMIT:
        raise ValueError("Set REPO_COMMIT to a full commit hash. Do not track a moving branch.")
    dest = Path("/content/long-context-attention-benchmark" if os.path.isdir("/content") else "colab_clone")
    if not (dest / ".git").exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(dest)])
    subprocess.check_call(["git", "checkout", REPO_COMMIT], cwd=dest)
    project = discover(dest)
else:
    project = Path(PROJECT_DIR).resolve() if PROJECT_DIR else discover(Path.cwd())

if project is None:
    raise FileNotFoundError(
        "No project source found. On a fresh Colab runtime, upload the project ZIP and set ZIP_PATH, "
        "or set REPO_URL and REPO_COMMIT."
    )
os.chdir(project)
if str(project) not in sys.path:
    sys.path.insert(0, str(project))
print("Project directory:", project)


In [ ]:
import importlib.util
bootstrap_path = project / "scripts" / "colab_bootstrap.py"
spec = importlib.util.spec_from_file_location("colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bootstrap)
bootstrap.install_project(project)
import spbench
print("spbench", spbench.__version__)


## Load runs and validate the schema


In [ ]:
import json
from pathlib import Path
import pandas as pd
from spbench.schema import read_jsonl

results_root = Path(RESULTS_DIR) if RESULTS_DIR else project / "results"
rows = []
for path in sorted(results_root.rglob("*.jsonl")):
    rows.extend(read_jsonl(path))
frame = pd.DataFrame(rows) if rows else pd.DataFrame()
print("jsonl rows", len(frame))
if frame.empty:
    print("No JSONL runs were found. Measured plots and the decision guide stay empty.")
else:
    print(frame.groupby(["origin", "status", "method"], dropna=False).size())
plan_path = results_root / "processed" / "dry_run_matrix.json"
if plan_path.is_file():
    plan = json.loads(plan_path.read_text())
    print("dry-run counts", plan["counts"])
    print("dry-run measured flag", plan["measured"])


## Measured plots, or an explicit skip


In [ ]:
from spbench.plots import plot_memory, plot_throughput
measured = pd.DataFrame()
if not frame.empty and "origin" in frame.columns:
    measured = frame[(frame["origin"] == "measured") & (frame["status"] == "success")].copy()
if measured.empty:
    print("No successful measured rows. Refusing to draw throughput or memory figures.")
else:
    comparable = measured.groupby(["dtype", "gpu_model", "topology"], dropna=False).size()
    print("Comparable groups (do not pool across these keys):")
    print(comparable)
    plot_throughput(measured, project / "results" / "figures" / "measured" / "throughput.png")
    plot_memory(measured, project / "results" / "figures" / "measured" / "memory.png")


## Decision guide status


In [ ]:
guide = project / "reports" / "generated" / "decision_guide_status.md"
guide.parent.mkdir(parents=True, exist_ok=True)
if measured.empty:
    text = (
        "# Decision guide status\n\n"
        "No successful measured distributed runs are loaded. "
        "A choice by sequence length, GPU count, or network cannot be made. "
        "Ring Attention, DistFlashAttn, DeepSpeed Ulysses, and USP remain untested on the required hardware.\n"
    )
else:
    text = (
        "# Decision guide status\n\n"
        f"Loaded {len(measured)} successful measured rows. "
        "Inspect dtype, GPU model, and topology before comparing them. "
        "This notebook does not name a universal winner.\n"
    )
guide.write_text(text)
print(guide.read_text())
